# KOSIS Chroma Coordinate Mapping Colab

이 노트북은 1차 `verification_input_ready` 또는 팀원이 준 IN-ready CSV를 받아서 KOSIS 2차 매핑을 실행합니다.

흐름:

```text
CSV 입력
→ 표 후보 hybrid search
→ period-aware table routing
→ KOSIS 메타 조회
→ Chroma coordinate 후보 검색
→ KOSIS API exact validation
→ mapping_status=READY만 실제값 검증
```

주의: API 키는 노트북에 저장하지 말고 실행 중 입력하세요.

In [ ]:
# 1. GitHub DBME 브랜치 clone
!rm -rf NLP_05-Team-Project-3
!git clone -b DBME https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git
%cd NLP_05-Team-Project-3

In [ ]:
# 2. 의존성 설치
!pip -q install -r requirements-ml.txt
!pip -q install python-dotenv requests pytest

In [ ]:
# 3. API 키 입력: 노트북 파일에 직접 쓰지 않기
import os
from getpass import getpass

if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = getpass('KOSIS_API_KEY: ')

print('KOSIS_API_KEY set:', bool(os.environ.get('KOSIS_API_KEY')))

In [ ]:
# 4. 입력 CSV 업로드
# 예: /Users/.../kosis_mapping_ready_500_real.csv 또는 팀원이 준 IN-ready CSV
from google.colab import files
uploaded = files.upload()
INPUT_CSV = next(iter(uploaded.keys()))
print('INPUT_CSV =', INPUT_CSV)

In [ ]:
# 5. 표 임베딩 인덱스 생성 또는 재생성
# 이미 data/indexes/kosis_bge_m3가 있으면 이 셀은 건너뛰어도 됩니다.
!python kosis_build_embedding_index.py \
  --table-index kosis_table_summary.csv \
  --out-dir data/indexes/kosis_bge_m3 \
  --model BAAI/bge-m3

In [ ]:
# 6. 빠른 1차 실행: 표 후보와 메타 후보 생성
# 전체 실행 전, 필요하면 --api-sample-limit을 작게 걸어 스모크 테스트하세요.
OUT_DIR = 'outputs/colab_chroma_run'
!rm -rf {OUT_DIR}
!python run_kosis_measurement_pipeline.py \
  --input {INPUT_CSV} \
  --table-index kosis_table_summary.csv \
  --out-dir {OUT_DIR} \
  --retrieval-mode hybrid \
  --top-tables 20 \
  --top-rank-for-meta 5 \
  --skip-meta

In [ ]:
# 7. 실제 메타 조회 + final candidates 생성
# 아직 값 검증은 하지 않고, coordinate DB를 만들 메타 파일을 먼저 생성합니다.
!python run_kosis_measurement_pipeline.py \
  --input {INPUT_CSV} \
  --table-index kosis_table_summary.csv \
  --out-dir {OUT_DIR} \
  --retrieval-mode hybrid \
  --top-tables 20 \
  --top-rank-for-meta 5 \
  --delay 0.12

In [ ]:
# 8. Chroma coordinate index 생성
from pathlib import Path

out = Path(OUT_DIR)
META_INDEX = str(next(out.glob('*_kosis_meta_index.csv')))
FINAL_CANDIDATES = str(next(out.glob('*_kosis_candidates_with_meta.csv')))
VALIDATED = str(out / (Path(INPUT_CSV).stem + '_kosis_validated_mappings.csv'))
VERIFIED = str(out / (Path(INPUT_CSV).stem + '_kosis_verified.csv'))
CHROMA_DIR = 'data/indexes/kosis_chroma_coordinates'

print('META_INDEX =', META_INDEX)
print('FINAL_CANDIDATES =', FINAL_CANDIDATES)

!python kosis_build_chroma_coordinate_index.py \
  --meta-index {META_INDEX} \
  --persist-dir {CHROMA_DIR} \
  --force

In [ ]:
# 9. Chroma coordinate 검색을 붙여 2차 READY 검증 실행
# READY가 된 행만 다음 셀에서 실제 수치 검증으로 넘어갑니다.
!python kosis_validate_mapping_candidates.py \
  --input {FINAL_CANDIDATES} \
  --meta-index {META_INDEX} \
  --output {VALIDATED} \
  --item-top-k 3 \
  --obj-top-k 2 \
  --api-candidate-top-k 5 \
  --chroma-coordinate-index {CHROMA_DIR} \
  --chroma-validation-top-k 20 \
  --delay 0.12

!python kosis_verify_claim_values.py \
  --input {VALIDATED} \
  --output {VERIFIED} \
  --delay 0.12

In [ ]:
# 10. 결과 요약
import csv
from pathlib import Path
from collections import Counter

out = Path(OUT_DIR)
for path in sorted(out.glob('*.csv')):
    print(path, path.stat().st_size)

validated = Path(VALIDATED) if Path(VALIDATED).exists() else None
verified = Path(VERIFIED) if Path(VERIFIED).exists() else None

if validated:
    rows = list(csv.DictReader(validated.open(encoding='utf-8-sig', newline='')))
    print('mapping_status:', Counter(r.get('mapping_status','') for r in rows))

if verified:
    rows = list(csv.DictReader(verified.open(encoding='utf-8-sig', newline='')))
    print('verdict:', Counter(r.get('verdict','') for r in rows))
    print('verdict_code:', Counter(r.get('verdict_code','') for r in rows))

In [ ]:
# 11. 결과 다운로드
from google.colab import files
import shutil

zip_path = shutil.make_archive('kosis_colab_results', 'zip', OUT_DIR)
files.download(zip_path)